## VARIABLES Y MÓDULOS

In [13]:
import sys, os
sys.path.append(os.path.abspath('..'))

from keras.datasets import mnist
from keras.utils import to_categorical
import numpy as np
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input, LeakyReLU
from keras import optimizers
from livelossplot import PlotLossesKeras
from livelossplot.outputs import MatplotlibPlot
from tensorflow.keras import callbacks
from tensorflow.keras.models import load_model
from Fuentes.imagen import DrawPanel
from PIL import Image, ImageOps
import glob
from sklearn.model_selection import train_test_split
from random import random
from scipy.ndimage import rotate


DATOS_DIR = '../Datos/'
FUENTES_DIR = '../Fuentes/'

## Ejercicio 1

La base de datos MNIST contiene imágenes de 28×28, en escala de grises, de números escritos a mano. Está conformada por 60.000 ejemplos de entrenamiento y 10.000 ejemplos de prueba.
Para cargar las imágenes utilice:

*from tensorflow.keras.datasets import mnist*

*(X_train, Y_train), (X_test, Y_test) = mnist.load_data()*

Puede visualizar una imagen utilizando:

*nImg = 0 # nro. de imagen a visualizar plt.imshow(X_train[0, :,:], cmap='gray')*

### a. Con el conjunto de 60000 imágenes entrene una red neuronal convolucional para predecir el dígito presente en la imagen. Recuerde normalizar los valores de cada imagen. Salve el modelo para recuperarlo después.

In [15]:
# Cargar y normalizar el conjunto de datos MNIST
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

Y_train= to_categorical(np.array(Y_train))
Y_test = to_categorical(np.array(Y_test))
IMG_SHAPE = X_train[0].shape
TARGET_CNT= len(Y_train[0])

print("Cantidad de imágenes de entrenamiento:", len(X_train))
print("Cantidad de imágenes de prueba:", len(X_test))
print("Forma de una imagen:", IMG_SHAPE)
print("Cantidad de clases objetivo:", TARGET_CNT)

X_train = X_train / 255
X_test  = X_test  / 255

X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test  = X_test.reshape(X_test.shape[0], 28, 28, 1)

Cantidad de imágenes de entrenamiento: 60000
Cantidad de imágenes de prueba: 10000
Forma de una imagen: (28, 28)
Cantidad de clases objetivo: 10


In [4]:
# %% Construccion del modelo
PADDING='same'
ACTIV='relu'

model = Sequential()

model.add(Input(shape=(*IMG_SHAPE, 1)))
model.add(Conv2D(32, kernel_size=(3,3), strides=(1,1), activation=ACTIV, padding=PADDING ))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(64, kernel_size=(3,3), strides=(1,1), activation=ACTIV, padding=PADDING ))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(128, kernel_size=(3,3), strides=(1,1), activation=ACTIV, padding=PADDING ))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(64, activation=ACTIV))
model.add(Dense(TARGET_CNT, activation='softmax'))

optimizer = optimizers.Adam(learning_rate=0.0001)
#optimizer = optimizers.RMSprop(learning_rate=0.0001)
#optimizer = optimizers.SGD(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'] )

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 28, 28, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 14, 14, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 7, 7, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 7, 7, 128)           │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 3, 3, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1152)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │          73,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 167,114 (652.79 KB)

 Trainable params: 167,114 (652.79 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
if not os.path.exists(FUENTES_DIR+'MNIST_conv_model.keras'):
    LOTES  = 128
    EPOCAS = 100
    PACIENCIA = 10
    
    # parada temprana para evitar el sobreajuste
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True )
    visual_plot = PlotLossesKeras( outputs=[ MatplotlibPlot(figsize=(12, 5)) ] )
    
    # %% Entrenamiento del modelo usando datos de entrenamiento y validacion
    H = model.fit(x=X_train, y=Y_train, batch_size=LOTES,
                  epochs=EPOCAS,
                  validation_split=0.3,
                  callbacks=[early_stop, visual_plot],
                  verbose=0
                  )

In [9]:
model.save(FUENTES_DIR+'MNIST_conv_model.keras')

### b. Levante el modelo guardado en el punto a) y utilice la clase DrawPanel del módulo utils.images de la carpeta fuentes para generar un dibujo escrito a mano de un dígito y predecir la clase a la que pertenece.

In [3]:
model = load_model(FUENTES_DIR+'MNIST_conv_model.keras')

In [7]:
IMG_SHAPE=(28,28)

drawn_image = DrawPanel(width=200, height=200)
drawn_image.show()

image = drawn_image.get_image()
image = image.resize((28, 28)).convert('L')  # Redimensionar y convertir a escala de grises
image_array = np.array(image) / 255.0  # Normalizar
image_array = image_array.reshape(1, 28, 28, 1)  # Ajustar la forma para el modelo
prediction = model.predict(image_array)
predicted_class = np.argmax(prediction)
print("El dígito predicho es:", predicted_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
El dígito predicho es: 5


## Ejercicio 2

Se buscará resolver la clasificación de los dígitos de MNIST usando la siguiente configuración:
- model = Sequential()
- model.add(Input(shape=(28, 28, 1)))
- model.add(Conv2D(F, kernel_size=K, strides=(S,S), activation=FUN))
- model.add(MaxPooling2D(pool_size=(2,2))) # -- opcional --
- model.add(Flatten())
- model.add(Dense(10,activation='softmax'))
- model.summary()

donde F es la cantidad de filtros o de mapas de características, K es el tamaño del kernel o máscara, S es el valor del stride y FUN es la función de activación de la capa de convolución.

La tabla que aparece a continuación sugiere los valores a utilizar. Se recomienda emplear Parada Temprana para reducir el tiempo de entrenamiento. 

Para ello utilice
- from tensorflow.keras.callbacks import EarlyStopping
- es = EarlyStopping(monitor='val_accuracy', patience=5, min_delta=0.001)

Esto indica que, si el valor del accuracy sobre los datos de validación no mejora después de 5 épocas, el entrenamiento finaliza. Puede usarse el parámetro min_delta para indicar cuando la diferencia entre dos accuracy se considerará significativa. Luego agregue este objeto en el momento del entrenamiento por medio del párametro callbacks

- H = model.fit(x = X_train, y = Y_train, batch_size = LOTES,
- validation_data = (X_test, Y_test), epochs=4000, callbacks=[es])

In [21]:
F = 4
K = (3,3)
S = 1
FUN = 'relu'
LOTES = 128

model = Sequential()
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(F, kernel_size=K, strides=(S,S), activation=FUN)) 
model.add(MaxPooling2D(pool_size=(2,2))) # -- opcional -- 
model.add(Flatten())
model.add(Dense(10,activation='softmax'))
optimizer = optimizers.Adam(learning_rate=0.0001)
#optimizer = optimizers.RMSprop(learning_rate=0.0001)
#optimizer = optimizers.SGD(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'] )
model.summary()

es = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, min_delta=0.001)

H = model.fit(x = X_train, y = Y_train, batch_size = LOTES,
validation_data = (X_test, Y_test), epochs=4000, callbacks=[es])

train_acc = H.history['accuracy']

epochs = range(1, len(train_acc)+1)

print('Epocas:',epochs)
print('Accuracy en train',train_acc)

test_loss, test_acc = model.evaluate(X_test, Y_test, batch_size=128)

print(f"Accuracy en test: {test_acc:.4f}")

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)                    │ (None, 26, 26, 4)           │              40 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 13, 13, 4)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_6 (Flatten)                  │ (None, 676)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 10)                  │           6,770 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,810 (26.60 KB)

 Trainable params: 6,810 (26.60 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4934 - loss: 2.0657 - val_accuracy: 0.7963 - val_loss: 1.6240
Epoch 2/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8236 - loss: 1.1193 - val_accuracy: 0.8602 - val_loss: 0.7326
Epoch 3/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8652 - loss: 0.6044 - val_accuracy: 0.8875 - val_loss: 0.4781
Epoch 4/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8881 - loss: 0.4449 - val_accuracy: 0.9047 - val_loss: 0.3818
Epoch 5/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9009 - loss: 0.3735 - val_accuracy: 0.9144 - val_loss: 0.3320
Epoch 6/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9096 - loss: 0.3322 - val_accuracy: 0.9188 - val_loss: 0.3009
Epoch 7/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9158 - loss: 0.3044 - val_accuracy: 0.9248 - val_loss: 0.2790
Epoch 8/4000
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9212 - loss: 0.2839 - 

## Ejercicio 3

Para resolver este ejercicio utilice un modelo de red neuronal convolucional que reconozca la cantidad de dedos extendidos en cada mano de las imágenes que conforman el juego de datos **“Fingers”**.

La versión original de este de estas imágenes se encuentra en https://www.kaggle.com/koryakinp/fingers.

Puede hallar una versión reducida de estas imágenes en el Moodle del curso, en la misma sección donde se encuentra este enunciado de práctica. También encontrará allí ejemplos sobre cómo cargar estas imágenes y cómo procesarlas con una red neuronal convolucional.
### a) Entrene y pruebe un modelo utilizando los datos de las carpetas **test** y **train**, midiendo **accuracy**

In [8]:
IMG_ERROR = 'No hay imágenes para cargar. Verificar que la ruta sea correcta y que la carpeta tenga imagenes con la extensión usada'

def import_data(data_dir):

    img_list = glob.glob(data_dir)      # Obtener la lista de archivos de imágenes

    assert len(img_list) > 0, IMG_ERROR # verifica que la ruta sea correcta y tenga al menos 1 imagen

    img_data = []  # lista de imagenes
    lbl_data = []  # lista de etiquetas

    img_count = len(img_list)
    for i, img_path in enumerate(img_list):

        img = Image.open(img_path)          # Carga imagen
        img = np.array(img) / np.max(img)   # Normaliza los píxeles entre 0 y 1
        img = img.reshape((*img.shape, 1))  # Formatea la imagen para TF: WxH => WxHx1

        # Almacenar la imagen y la etiqueta
        img_data.append(img)
        # ej. nombre de archivo: 000e7aa6-100b-4c6b-9ff0-e7a8e53e4465_5L.png
        lbl_data.append(int(img_path[-6]))  # Extrae la cantidad de dedos del nombre del archivo

        # Mostrar progreso en la carga
        if i % 100 == 0:
            print("\rCargando imágenes: %6.2f%%" % (100 * i / img_count), end="")

    print("\rCargando imágenes: 100.00%% (%d) \n" % img_count)

    return np.array(img_data), np.array(lbl_data)


# carga las imagenes a partir de los nombres de archivos
x_train, y_train = import_data(DATOS_DIR+"Fingers/train/*/*.png")

# carga las imagenes a partir de los nombres de archivos
x_test, y_test = import_data(DATOS_DIR+"Fingers/test/*/*.png")

# separa los datos y clase en grupo de entrenamiento y validacion
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size = 0.30, shuffle = True)


Cargando imágenes: 100.00% (17999) 

Cargando imágenes: 100.00% (3600) 



In [11]:
EPOCAS = 100
LOTES  = 128
PACIENCIA=5
IMG_SIZE = x_train.shape[1:]
N_CLASSES = len(np.unique(y_train))
ACTIVA =LeakyReLU()

# Construye el modelo
model = Sequential()

model.add(Input( shape=IMG_SIZE ))
model.add(Conv2D(16, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(32, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(32, activation = ACTIVA))
model.add(Dense(N_CLASSES, activation = 'softmax'))

model.summary()

# construye el modelo
optimizer = optimizers.Adam(0.001)
# Observar que con "sparse_categorical_crossentropy" no hace falta codificacion one-hot para las clases
model.compile(optimizer, loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])


# parada temprana para evitar el sobreajuste
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, min_delta=0.001, restore_best_weights=True )

# entrena el modelo y guarda la historia del progreso
H = model.fit(x_train,
              y_train,
              batch_size = LOTES,
              epochs = EPOCAS,
              validation_data = (x_val, y_val),
              callbacks=[early_stop]
             )

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 64, 64, 16)          │             160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 32, 32, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 32, 32, 32)          │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 16, 16, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 8192)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │         262,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 6)                   │             198 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 267,174 (1.02 MB)

 Trainable params: 267,174 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 12s 92ms/step - accuracy: 0.8917 - loss: 0.3755 - val_accuracy: 0.9896 - val_loss: 0.0363
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 10s 90ms/step - accuracy: 0.9947 - loss: 0.0238 - val_accuracy: 0.9998 - val_loss: 0.0066
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 9s 89ms/step - accuracy: 0.9997 - loss: 0.0047 - val_accuracy: 0.9998 - val_loss: 0.0026
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 9s 88ms/step - accuracy: 1.0000 - loss: 0.0018 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 10s 99ms/step - accuracy: 1.0000 - loss: 9.4888e-04 - val_accuracy: 1.0000 - val_loss: 7.4584e-04
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 10s 94ms/step - accuracy: 1.0000 - loss: 5.6936e-04 - val_accuracy: 1.0000 - val_loss: 5.3344e-04
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - accuracy: 1.0000 - loss: 3.5477e-04 - val_accuracy: 0.9998 - val_loss: 5.1460e-04
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 9s 96ms/step - accuracy: 1.0000 - lo

In [12]:
pred_train = model.evaluate(x_train, y_train, verbose=0)
pred_test = model.evaluate(x_test, y_test, verbose=0)
print("\nEfectividad del modelo con datos de entrenamiento: %6.2f%%" % (H.history['accuracy'][-1]*100))
print("Efectividad del modelo con datos de validacion...: %6.2f%%" % (H.history['val_accuracy'][-1]*100))
print("Efectividad del modelo con datos de Prueba.......: %6.2f%%" % (pred_test[1]*100))


Efectividad del modelo con datos de entrenamiento: 100.00%
Efectividad del modelo con datos de validacion...: 100.00%
Efectividad del modelo con datos de Prueba.......:  99.94%


### b) Genere una versión del **dataset** para **test** agregando transformaciones al azar sobres imágenes originales. Haga **rotaciones** entre -45 y 45 grados, repita el test y mida el accuracy.

In [18]:
def rotar_al_azar(data_imgs, max_ang):
    result = np.empty_like(data_imgs)
    for i, img in enumerate(data_imgs):
        ang = (random() - 0.5) * 2 * max_ang #random genera nros aleatorios entre 0 y 1
        result[i] = rotate(img, ang, reshape=False, mode='reflect')
    return result

x_test_rot = rotar_al_azar(x_test, 45)

# evalua el modelo con los datos de testeo
pred = model.evaluate(x_test_rot, y_test, verbose=0)
print("Efectividad del modelo con datos de Prueba.......: %6.2f%%" % (pred[1]*100))

Efectividad del modelo con datos de Prueba.......:  71.03%


### c) Genere una versión del **dataset train** como la realizada en b) y repita entrenamiento y prueba de a) con los datasets de modificados.

In [19]:
x_train_rot = rotar_al_azar(x_train, 45)

EPOCAS = 100
LOTES  = 128
PACIENCIA=5
IMG_SIZE = x_train.shape[1:]
N_CLASSES = len(np.unique(y_train))
ACTIVA =LeakyReLU()

# Construye el modelo
model = Sequential()

model.add(Input( shape=IMG_SIZE ))
model.add(Conv2D(16, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(32, kernel_size=(3,3), strides=(1,1), padding='same', activation=ACTIVA))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(32, activation = ACTIVA))
model.add(Dense(N_CLASSES, activation = 'softmax'))

model.summary()

# construye el modelo
optimizer = optimizers.Adam(0.001)
# Observar que con "sparse_categorical_crossentropy" no hace falta codificacion one-hot para las clases
model.compile(optimizer, loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])


# parada temprana para evitar el sobreajuste
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, min_delta=0.001, restore_best_weights=True )

# entrena el modelo y guarda la historia del progreso
H = model.fit(x_train_rot,
              y_train,
              batch_size = LOTES,
              epochs = EPOCAS,
              validation_data = (x_val, y_val),
              callbacks=[early_stop]
             )

pred_train = model.evaluate(x_train_rot, y_train, verbose=0)
pred_test = model.evaluate(x_test_rot, y_test, verbose=0)
print("\nEfectividad del modelo con datos de entrenamiento: %6.2f%%" % (H.history['accuracy'][-1]*100))
print("Efectividad del modelo con datos de validacion...: %6.2f%%" % (H.history['val_accuracy'][-1]*100))
print("Efectividad del modelo con datos de Prueba.......: %6.2f%%" % (pred_test[1]*100))

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)                    │ (None, 64, 64, 16)          │             160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 32, 32, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 32, 32, 32)          │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 16, 16, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 8192)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │         262,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 6)                   │             198 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 267,174 (1.02 MB)

 Trainable params: 267,174 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 11s 94ms/step - accuracy: 0.7176 - loss: 0.7742 - val_accuracy: 0.8987 - val_loss: 0.2722
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 10s 91ms/step - accuracy: 0.9694 - loss: 0.1173 - val_accuracy: 0.9863 - val_loss: 0.0593
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 9s 90ms/step - accuracy: 0.9859 - loss: 0.0525 - val_accuracy: 0.9937 - val_loss: 0.0335
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 12s 118ms/step - accuracy: 0.9952 - loss: 0.0251 - val_accuracy: 0.9948 - val_loss: 0.0245
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 23s 141ms/step - accuracy: 0.9964 - loss: 0.0171 - val_accuracy: 0.9985 - val_loss: 0.0115
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 15s 152ms/step - accuracy: 0.9976 - loss: 0.0126 - val_accuracy: 0.9994 - val_loss: 0.0066
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 14s 141ms/step - accuracy: 0.9986 - loss: 0.0080 - val_accuracy: 0.9983 - val_loss: 0.0122
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 14s 139ms/step - accuracy: 0.9994 - loss: 0.0048 - val_